# Trabajo Práctico N° 5 - Contadores
## Medidas Electrónicas I

**Docentes:**

*   Ing. Marinsek, Emiliano
*   Ing. Perdomo, Juan Manuel

**Alumnos:**

*   Ferrario, Franco Ezequiel
*   Encinas, Leandro
*   Marchesi, Matias
*   Quiroga, Bruno

**Grupo:** 2

**Curso:** R4052

**Fecha de realización:** 24/06/2025

**Año:** 2026

# Librerias y funciones a utilizar

In [1]:
# Librerias de python a utilizar
import numpy as np
import math
from IPython.display import display, Math
import pandas as pd


def incert_A(mediciones):
  s = np.std(mediciones, ddof=1)  # ddof=1 -> grados de libertad v = n-1
  return s/math.sqrt(len(mediciones))

def incert_B_ConMedia(err_rel, nd, ctas, media):
  # Distribución rectangular: (error relativo [%] + nd/cuentas) * media / sqrt(3)
  return (err_rel/100 + nd/ctas) * media / math.sqrt(3)

def incert_Total(uA, uB):
  return math.sqrt(uA**2 + uB**2)

def incert_Comb_Rel(uX, uY, correlacion=0):
  return math.sqrt((uX['u_C']/uX['media'])**2 + (uY['u_C']/uY['media'])**2 - (2*(uX['u_C']/uX['media'])*(uY['u_C']/uY['media'])*correlacion))

def redondear(media, U):
  """Redondea U a 2 cifras significativas y acota la media a los mismos dígitos."""
  e = math.floor(math.log10(abs(U))) - 1
  U_r = round(U / 10**e) * 10**e
  if U_r >= 10**(e + 2):          # p.ej. 0.0996 -> 0.10
    e += 1
    U_r = round(U / 10**e) * 10**e
  dec = max(-e, 0)
  m_r = round(media / 10**e) * 10**e
  return f"{m_r:.{dec}f}", f"{U_r:.{dec}f}"

def imprimir(nombre, unidad, media, uA, uB, uC, U):
  m, u = redondear(media, U)
  print(f"Media = {media:.10g} {unidad}")
  print(f"uA = {uA:.3e} {unidad}")
  print(f"uB = {uB:.3e} {unidad}")
  print(f"uC = {uC:.3e} {unidad}")
  print(f"U (k={k}) = {U:.3e} {unidad}")
  print(f"Resultado: {nombre} = ({m} ± {u}) {unidad}")

def medir_frecuencia(mediciones, gate_time):
  """Modo frecuencímetro: N = f_x * T_C ; Δf/f = ΔfBT/fBT + ΔN/N"""
  f_media = np.mean(mediciones)
  N = f_media * gate_time                       # cuentas durante el gate time
  uA = incert_A(mediciones)
  uB = incert_B_ConMedia(error_rel_BT_pct, nd, N, f_media)
  uC = incert_Total(uA, uB)
  U = k * uC
  print(f"N = {N:.6g} cuentas")
  imprimir("f", "Hz", f_media, uA, uB, uC, U)
  return {'media': f_media, 'u_C': uC, 'U': U}

def medir_periodo(mediciones_us, M):
  """Modo período promediado: N = M * T_x * f_BT ; ΔT/T = ΔfBT/fBT + ΔN/N (e_D = 0, señal cuadrada)"""
  T_media = np.mean(mediciones_us)              # [µs]
  N = M * T_media * 1e-6 * f_BT                 # pulsos de reloj contados en M períodos
  uA = incert_A(mediciones_us)
  uB = incert_B_ConMedia(error_rel_BT_pct, nd, N, T_media)
  uC = incert_Total(uA, uB)
  U = k * uC
  print(f"N = {N:.6g} cuentas  ->  resolución = {T_media/N:.3g} µs")
  imprimir("T", "µs", T_media, uA, uB, uC, U)
  return {'media': T_media, 'u_C': uC, 'U': U}

resultados = []   # tabla resumen al final del informe

# Generalidades de los instrumentos

El frecuencímetro utilizado para las prácticas es el contador universal **Protek (Hung Chang) U2000A** ([hoja de datos](recursos/Protek_9100_U2000A.pdf)). Sus especificaciones relevantes son:

* Base de tiempo: oscilador de referencia de $f_{BT} = 10\,\text{MHz}$.
* Display de 8 dígitos.
* Gate time (modo frecuencia): 0.01 / 0.1 / 1 / 10 s.
* Multiplicador (períodos promediados, T.I. A→B y Ratio A/B): x1 / x10 / x100 / x1000.

Para el cálculo de incertidumbres se utilizará la estabilidad de largo plazo de la base de tiempo especificada por el fabricante, cuyo valor es:

$$
\frac{\Delta f_{BT}}{f_{BT}} = \pm 3\cdot10^{-7}/\text{mes}
$$

Como no se dispone de información sobre la antigüedad del instrumento, se adopta como hipótesis una antigüedad de **5 años**, equivalentes a 60 meses. Por lo tanto:

$$
\frac{\Delta f_{BT}}{f_{BT}} = 3\cdot10^{-7}\cdot\frac{1}{\text{mes}}\cdot60\,\text{meses}
$$

Expresando este valor en partes por millón:

$$
\frac{\Delta f_{BT}}{f_{BT}} = 18\,\text{ppm}
$$

y en porcentaje:

$$
\frac{\Delta f_{BT}}{f_{BT}} = 18\cdot10^{-6} \cdot 100\% = 0.0018\%
$$

Este valor será utilizado como error relativo de la base de tiempo en los cálculos de incertidumbre. Todas las incertidumbres Tipo B se modelan con distribución rectangular (división por $\sqrt{3}$) y la incertidumbre expandida se calcula con un factor de cobertura $k = 2$ (nivel de confianza de aproximadamente 95%).

A continuación se coloca una celda con las variables generales del instrumento que se usan en todas las demás celdas.

In [2]:
k = 2

f_BT = 10e6                 # Frecuencia de la base de tiempo [Hz]

estabilidad_BT = 3e-7       # Estabilidad de largo plazo [1/mes]
antiguedad_meses = 5 * 12   # Antigüedad estimada del instrumento [meses]

error_rel_BT = estabilidad_BT * antiguedad_meses
error_rel_BT_pct = error_rel_BT * 100

nd = 1                      # Error de ±1 cuenta

print(f"Base de tiempo: {f_BT/1e6} MHz")
print(f"Antigüedad estimada: {antiguedad_meses} meses")
print(f"Estabilidad acumulada: {error_rel_BT:.2e}")
print(f"Error relativo de BT: {error_rel_BT_pct:.4f} %")

Base de tiempo: 10.0 MHz
Antigüedad estimada: 60 meses
Estabilidad acumulada: 1.80e-05
Error relativo de BT: 0.0018 %


# Procedimiento de medición

**Instrumental utilizado:**

* Contador universal Protek U2000A.
* Generador de funciones (señales cuadradas, rectangulares de ciclo de actividad variable y senoidales).
* Osciloscopio digital, para verificar la forma de onda, la amplitud y el ciclo de actividad de las señales.
* Cables coaxiales BNC y una "T" BNC.

**Conexión:** la salida del generador se conecta al canal A del contador y, en paralelo con una "T" BNC, al osciloscopio. En la medición de intervalos (T.I. A→B) la misma señal se lleva a los canales A y B mediante la "T", y se elige con *SLOPE* el flanco de disparo de cada canal. En el modo Ratio A/B se conecta una señal en cada canal.

<img src="recursos/TP5.jpeg" width="350">

*Contador durante la medición T.I. A→B con multiplicador x100: la misma señal ingresa a los canales A y B mediante una "T" BNC.*

**Pasos:**

1. Configurar el generador con una señal cuadrada de $2\,V_{pp}$ a la frecuencia a medir y verificarla en el osciloscopio.
2. Modo frecuencia (FREQ A): seleccionar gate time de 0.1 s, 1 s y 10 s y tomar 10 lecturas para cada uno.
3. Modo período (PERI. A): seleccionar el multiplicador x1, x10 y x100 (cantidad de períodos promediados) y tomar 10 lecturas para cada uno.
4. Repetir para 10 Hz, 1 kHz y 1 MHz.
5. Modo T.I. A→B con multiplicador x100: con la señal rectangular de 1 kHz y ciclos de actividad de 20%, 50% y 80%, medir las cuatro combinaciones de flancos (++, +-, -+, --) y tomar 10 lecturas de cada una.
6. Modo Ratio A/B con multiplicador x1000 (máxima resolución): conectar las senoidales de 10 MHz y 1 MHz y tomar 10 lecturas.

# Ejercicio 1

Las ecuaciones que se usarán para este ejercicio son las vistas en la cátedra.

## Medición de frecuencia

El contador cuenta los $N$ ciclos de la señal que entran durante el tiempo de compuerta (gate time) $T_C$, generado a partir de la base de tiempo:

$$
f_x = \frac{N}{T_C}
$$

Por lo tanto, la cantidad de ciclos durante el gate time es:

$$
N = f_x \cdot T_C
$$

y la incertidumbre relativa es:

$$
\frac{\Delta f_x}{f_x} = \pm \left(\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N}\right)
$$

siendo $\Delta N = 1$ cuenta. La resolución del display es $1/T_C$: 10 Hz con 0.1 s, 1 Hz con 1 s y 0.1 Hz con 10 s.

## Medición de períodos promediados

El contador cuenta los $N$ pulsos de la base de tiempo ($T_{BT} = 1/f_{BT} = 0.1\,\mu\text{s}$) durante $M$ períodos de la señal:

$$
T_x = \frac{N \cdot T_{BT}}{M} \qquad \Rightarrow \qquad N = M \cdot T_x \cdot f_{BT}
$$

y su incertidumbre relativa es:

$$
\frac{\Delta T_x}{T_x} = \pm \left(\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N} + \frac{e_D}{M \cdot T_x}\right)
$$

donde $M$ es la cantidad de períodos promediados (1, 10 o 100) y $e_D$ es el error de disparo. Como las señales son cuadradas, de flancos muy rápidos, $e_D \approx 0$ (se analiza en el Extra), y la incertidumbre queda:

$$
\frac{\Delta T_x}{T_x} = \pm \left(\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N}\right)
$$

La resolución del display es $T_{BT}/M$: 0.1 µs, 0.01 µs y 0.001 µs para $M$ = 1, 10 y 100.

En ambos modos se evalúa la incertidumbre Tipo B con `incert_B_ConMedia(error_rel_BT_pct, nd, N, media)` y se combina con la Tipo A.

## 10 Hz

Se medirá una señal cuadrada de $10\,\text{Hz}$ con $2\,V_{pp}$.

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de gate time.

In [3]:
# Datos 0.1s de Gate Time
mediciones = [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]  # Hz

f_10_01 = medir_frecuencia(mediciones, gate_time=0.1)

N = 1 cuentas
Media = 10 Hz
uA = 0.000e+00 Hz
uB = 5.774e+00 Hz
uC = 5.774e+00 Hz
U (k=2) = 1.155e+01 Hz
Resultado: f = (10 ± 12) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (10 \pm 12)\,\text{Hz}}
$$

### 1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de gate time.

In [4]:
# Datos 1s de Gate Time
mediciones = [10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]  # Hz

f_10_1 = medir_frecuencia(mediciones, gate_time=1)

N = 10 cuentas
Media = 10 Hz
uA = 0.000e+00 Hz
uB = 5.775e-01 Hz
uC = 5.775e-01 Hz
U (k=2) = 1.155e+00 Hz
Resultado: f = (10.0 ± 1.2) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (10.0 \pm 1.2)\,\text{Hz}}
$$

### 10s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de gate time.

In [5]:
# Datos 10s de Gate Time
mediciones = [10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.00]  # Hz

f_10_10 = medir_frecuencia(mediciones, gate_time=10)

N = 100 cuentas
Media = 10 Hz
uA = 0.000e+00 Hz
uB = 5.784e-02 Hz
uC = 5.784e-02 Hz
U (k=2) = 1.157e-01 Hz
Resultado: f = (10.00 ± 0.12) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (10.00 \pm 0.12)\,\text{Hz}}
$$

### 1 ciclo de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de períodos promediados (multiplicador x1).

In [6]:
# Datos período promediado M = 1
mediciones = [100003.0, 100002.9, 100003.0, 100002.9, 100003.0, 100002.9, 100003.0, 100002.9, 100003.0, 100002.9]  # µs

T_10_1 = medir_periodo(mediciones, M=1)

N = 1.00003e+06 cuentas  ->  resolución = 0.1 µs
Media = 100002.95 µs
uA = 1.667e-02 µs
uB = 1.097e+00 µs
uC = 1.097e+00 µs
U (k=2) = 2.194e+00 µs
Resultado: T = (100002.9 ± 2.2) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (100002.9 \pm 2.2)\,\mu\text{s}}
$$

### 10 ciclos de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclos de períodos promediados (multiplicador x10).

In [7]:
# Datos período promediado M = 10
mediciones = [100003.06, 100003.09, 100003.13, 100003.16, 100003.24,
               100003.28, 100003.27, 100003.29, 100003.30, 100003.29]  # µs

T_10_10 = medir_periodo(mediciones, M=10)

N = 1.00003e+07 cuentas  ->  resolución = 0.01 µs
Media = 100003.211 µs
uA = 2.908e-02 µs
uB = 1.045e+00 µs
uC = 1.045e+00 µs
U (k=2) = 2.091e+00 µs
Resultado: T = (100003.2 ± 2.1) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (100003.2 \pm 2.1)\,\mu\text{s}}
$$

### 100 ciclos de períodos promediados

No se pudo realizar la medición porque el instrumento indicó **OVERFLOW**. Con $M = 100$ la resolución es de 0.001 µs y el período de la señal es de unos 100003 µs, de modo que la lectura necesitaría $100003.000\,\mu\text{s}$, es decir 9 dígitos, uno más de los 8 que tiene el display. Además, cada lectura requiere $100 \cdot 0.1\,\text{s} = 10\,\text{s}$ de medición. Las pocas lecturas que se registraron (3.468, 3.475 y 3.480 µs) corresponden a los dígitos menos significativos con el desborde activado y no representan el período, por lo que no se informa un resultado.

## 1 kHz

Se medirá una señal cuadrada de $1\,\text{kHz}$ con $2\,V_{pp}$.

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de gate time.

In [8]:
# Datos 0.1s de Gate Time
mediciones = [1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3, 1.00e3]  # Hz

f_1k_01 = medir_frecuencia(mediciones, gate_time=0.1)

N = 100 cuentas
Media = 1000 Hz
uA = 0.000e+00 Hz
uB = 5.784e+00 Hz
uC = 5.784e+00 Hz
U (k=2) = 1.157e+01 Hz
Resultado: f = (1000 ± 12) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (1000 \pm 12)\,\text{Hz}}
$$

### 1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de gate time.

In [9]:
# Datos 1s de Gate Time
mediciones = [1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3, 1.000e3]  # Hz

f_1k_1 = medir_frecuencia(mediciones, gate_time=1)

N = 1000 cuentas
Media = 1000 Hz
uA = 0.000e+00 Hz
uB = 5.877e-01 Hz
uC = 5.877e-01 Hz
U (k=2) = 1.175e+00 Hz
Resultado: f = (1000.0 ± 1.2) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (1000.0 \pm 1.2)\,\text{Hz}}
$$

### 10s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de gate time.

In [10]:
# Datos 10s de Gate Time
mediciones = [1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3, 1.0000e3]  # Hz

f_1k_10 = medir_frecuencia(mediciones, gate_time=10)

N = 10000 cuentas
Media = 1000 Hz
uA = 0.000e+00 Hz
uB = 6.813e-02 Hz
uC = 6.813e-02 Hz
U (k=2) = 1.363e-01 Hz
Resultado: f = (1000.00 ± 0.14) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (1000.00 \pm 0.14)\,\text{Hz}}
$$

### 1 ciclo de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de períodos promediados (multiplicador x1).

In [11]:
# Datos período promediado M = 1
mediciones = [1000.0, 1000.1, 1000.0, 1000.1, 1000.0, 1000.1, 1000.0, 1000.1, 1000.0, 1000.1]  # µs

T_1k_1 = medir_periodo(mediciones, M=1)

N = 10000.5 cuentas  ->  resolución = 0.1 µs
Media = 1000.05 µs
uA = 1.667e-02 µs
uB = 6.813e-02 µs
uC = 7.014e-02 µs
U (k=2) = 1.403e-01 µs
Resultado: T = (1000.05 ± 0.14) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1000.05 \pm 0.14)\,\mu\text{s}}
$$

### 10 ciclos de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclos de períodos promediados (multiplicador x10).

In [12]:
# Datos período promediado M = 10
mediciones = [1000.03, 1000.04, 1000.03, 1000.04, 1000.03, 1000.04, 1000.03, 1000.04, 1000.03, 1000.04]  # µs

T_1k_10 = medir_periodo(mediciones, M=10)

N = 100003 cuentas  ->  resolución = 0.01 µs
Media = 1000.035 µs
uA = 1.667e-03 µs
uB = 1.617e-02 µs
uC = 1.625e-02 µs
U (k=2) = 3.250e-02 µs
Resultado: T = (1000.035 ± 0.033) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1000.035 \pm 0.033)\,\mu\text{s}}
$$

### 100 ciclos de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 100 ciclos de períodos promediados (multiplicador x100).

In [13]:
# Datos período promediado M = 100
mediciones = [1000.035, 1000.036, 1000.035, 1000.036, 1000.035, 1000.036, 1000.035, 1000.036, 1000.035, 1000.036]  # µs

T_1k_100 = medir_periodo(mediciones, M=100)

N = 1.00004e+06 cuentas  ->  resolución = 0.001 µs
Media = 1000.0355 µs
uA = 1.667e-04 µs
uB = 1.097e-02 µs
uC = 1.097e-02 µs
U (k=2) = 2.194e-02 µs
Resultado: T = (1000.035 ± 0.022) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1000.035 \pm 0.022)\,\mu\text{s}}
$$

## 1 MHz

Se medirá una señal cuadrada de $1\,\text{MHz}$ con $2\,V_{pp}$.

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de gate time.

In [14]:
# Datos 0.1s de Gate Time
mediciones = [999.96e3, 999.97e3, 999.96e3, 999.97e3, 999.96e3, 999.97e3, 999.96e3, 999.97e3, 999.96e3, 999.97e3]  # Hz

f_1M_01 = medir_frecuencia(mediciones, gate_time=0.1)

N = 99996.5 cuentas
Media = 999965 Hz
uA = 1.667e+00 Hz
uB = 1.617e+01 Hz
uC = 1.625e+01 Hz
U (k=2) = 3.250e+01 Hz
Resultado: f = (999965 ± 33) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (999965 \pm 33)\,\text{Hz}}
$$

### 1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de gate time.

In [15]:
# Datos 1s de Gate Time
mediciones = [999.965e3, 999.964e3, 999.964e3, 999.965e3, 999.965e3,
              999.965e3, 999.965e3, 999.965e3, 999.964e3, 999.964e3]  # Hz

f_1M_1 = medir_frecuencia(mediciones, gate_time=1)

N = 999965 cuentas
Media = 999964.6 Hz
uA = 1.633e-01 Hz
uB = 1.097e+01 Hz
uC = 1.097e+01 Hz
U (k=2) = 2.194e+01 Hz
Resultado: f = (999965 ± 22) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (999965 \pm 22)\,\text{Hz}}
$$

### 10s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de gate time.

In [16]:
# Datos 10s de Gate Time
mediciones = [999.9648e3, 999.9648e3, 999.9648e3, 999.9647e3, 999.9648e3,
              999.9648e3, 999.9648e3, 999.9647e3, 999.9648e3, 999.9648e3]  # Hz

f_1M_10 = medir_frecuencia(mediciones, gate_time=10)

N = 9.99965e+06 cuentas
Media = 999964.78 Hz
uA = 1.333e-02 Hz
uB = 1.045e+01 Hz
uC = 1.045e+01 Hz
U (k=2) = 2.090e+01 Hz
Resultado: f = (999965 ± 21) Hz


Finalmente se expresa el resultado:

$$
\boxed{f = (999965 \pm 21)\,\text{Hz}}
$$

### 1 ciclo de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de períodos promediados (multiplicador x1).

In [17]:
# Datos período promediado M = 1
mediciones = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]  # µs

T_1M_1 = medir_periodo(mediciones, M=1)

N = 10 cuentas  ->  resolución = 0.1 µs
Media = 1 µs
uA = 0.000e+00 µs
uB = 5.775e-02 µs
uC = 5.775e-02 µs
U (k=2) = 1.155e-01 µs
Resultado: T = (1.00 ± 0.12) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1.00 \pm 0.12)\,\mu\text{s}}
$$

### 10 ciclos de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclos de períodos promediados (multiplicador x10).

In [18]:
# Datos período promediado M = 10
mediciones = [1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00]  # µs

T_1M_10 = medir_periodo(mediciones, M=10)

N = 100 cuentas  ->  resolución = 0.01 µs
Media = 1 µs
uA = 0.000e+00 µs
uB = 5.784e-03 µs
uC = 5.784e-03 µs
U (k=2) = 1.157e-02 µs
Resultado: T = (1.000 ± 0.012) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1.000 \pm 0.012)\,\mu\text{s}}
$$

### 100 ciclos de períodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 100 ciclos de períodos promediados (multiplicador x100).

In [19]:
# Datos período promediado M = 100
mediciones = [1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000]  # µs

T_1M_100 = medir_periodo(mediciones, M=100)

N = 1000 cuentas  ->  resolución = 0.001 µs
Media = 1 µs
uA = 0.000e+00 µs
uB = 5.877e-04 µs
uC = 5.877e-04 µs
U (k=2) = 1.175e-03 µs
Resultado: T = (1.0000 ± 0.0012) µs


Finalmente se expresa el resultado:

$$
\boxed{T = (1.0000 \pm 0.0012)\,\mu\text{s}}
$$

## Extra: estabilidad de la indicación con señal cuadrada y senoidal

Al cambiar la señal cuadrada por una senoidal de igual frecuencia y amplitud, la indicación se vuelve **menos estable**: los últimos dígitos fluctúan más entre lecturas, sobre todo en modo período y a baja frecuencia. La causa principal es el **error de disparo** $e_D$.

El contador abre y cierra la compuerta cuando la señal cruza el nivel de disparo (*trigger level*) de su comparador de entrada. Si la señal tiene un ruido superpuesto de valor $V_n$, el instante de cruce se adelanta o se retrasa en:

$$
\Delta t = \frac{V_n}{\left. \dfrac{dv}{dt} \right|_{cruce}}
$$

* **Señal cuadrada:** el flanco es casi vertical. Con un tiempo de subida $t_r$, la pendiente es $\dfrac{dv}{dt} \approx \dfrac{V_{pp}}{t_r}$, muy grande, por lo que $\Delta t$ es despreciable.
* **Señal senoidal:** $v(t) = V_p \sin(2\pi f t)$, y la pendiente máxima (en el cruce por cero) es $\dfrac{dv}{dt} = 2\pi f V_p$, mucho menor, y más aún a baja frecuencia.

Como el error puede ocurrir en el flanco de apertura y en el de cierre, $e_D \approx 2\Delta t$, y el error relativo en el período promediado es:

$$
\frac{e_D}{M\,T_x} = \frac{2 V_n}{2\pi f V_p}\cdot\frac{f}{M} = \frac{V_n}{\pi V_p M}
$$

Para la senoidal este error relativo no depende de la frecuencia: su valor absoluto ($e_D$) crece al bajar $f$, pero el período crece en la misma proporción. Solo se reduce promediando más períodos ($M$) o aumentando la amplitud, es decir, la pendiente.

En modo frecuencia el efecto aparece de otra forma: si el ruido supera la histéresis del comparador cerca del cruce lento, puede generar cuentas espurias y la cuenta $N$ deja de ser estable.

La siguiente celda estima el orden de magnitud con un ruido supuesto de $V_n = 1\,\text{mV}$, $V_p = 1\,\text{V}$ (señal de 2 Vpp) y $t_r = 20\,\text{ns}$ para la cuadrada.

In [20]:
V_n = 1e-3      # ruido supuesto [V]
V_p = 1.0       # amplitud pico (2 Vpp) [V]
t_r = 20e-9     # tiempo de subida supuesto de la cuadrada [s]

for f in [10, 1e3, 1e6]:
    eD_sen = 2 * V_n / (2*np.pi*f*V_p)          # error de disparo senoidal [s]
    eD_cuad = 2 * V_n / (2*V_p / t_r)           # error de disparo cuadrada [s]
    T = 1/f
    print(f"f = {f:>9.0f} Hz | e_D senoidal = {eD_sen:.2e} s ({eD_sen/T*1e6:7.1f} ppm de T)"
          f" | e_D cuadrada = {eD_cuad:.2e} s ({eD_cuad/T*1e6:.2e} ppm de T)")

f =        10 Hz | e_D senoidal = 3.18e-05 s (  318.3 ppm de T) | e_D cuadrada = 2.00e-11 s (2.00e-04 ppm de T)
f =      1000 Hz | e_D senoidal = 3.18e-07 s (  318.3 ppm de T) | e_D cuadrada = 2.00e-11 s (2.00e-02 ppm de T)
f =   1000000 Hz | e_D senoidal = 3.18e-10 s (  318.3 ppm de T) | e_D cuadrada = 2.00e-11 s (2.00e+01 ppm de T)


Con una senoidal, el error de disparo es del orden de **300 ppm por período (M = 1)**, muy superior a los 18 ppm de la base de tiempo, mientras que con la cuadrada es despreciable salvo a 1 MHz. Esto explica por qué con señal senoidal la indicación es mucho menos estable y por qué para la medición de período conviene usar señales de flancos rápidos, o bien aumentar la amplitud y la cantidad de períodos promediados.

# Ejercicio 2

Se mide el ciclo de actividad y el período de una serie de señales rectangulares de 1 kHz con un 20%, 50% y 80% de ciclo de actividad, utilizando el modo de medición de intervalos (T.I. A→B). La misma señal ingresa a los canales A y B: el canal A abre la compuerta y el canal B la cierra.

### Definición de Intervalos de Tiempo (T.I. A $\rightarrow$ B)

Según la combinación de flancos configurada (*SLOPE* de cada canal), el intervalo medido es:

* **$(+, +)$**: de un flanco ascendente al siguiente flanco ascendente. Representa el **período ($T$)**.
* **$(-, -)$**: de un flanco descendente al siguiente flanco descendente. Representa también el **período ($T$)**.
* **$(+, -)$**: de un flanco ascendente al flanco descendente siguiente. Representa el **ancho de pulso positivo ($\tau_+$)**, es decir, el tiempo en estado alto.
* **$(-, +)$**: de un flanco descendente al flanco ascendente siguiente. Representa el **ancho de pulso negativo ($\tau_-$)**, es decir, el tiempo en estado bajo.

Las mediciones se realizaron con el **multiplicador x100**: el contador promedia 100 intervalos y cuenta pulsos de la base de tiempo, por lo que

$$
\tau = \frac{N \cdot T_{BT}}{100} \qquad \Rightarrow \qquad N = 100 \cdot \tau \cdot f_{BT}
$$

con una resolución de $0.1\,\mu\text{s}/100 = 1\,\text{ns}$. La incertidumbre de cada intervalo es la misma que la de período:

$$
\frac{\Delta \tau}{\tau} = \pm \left(\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N}\right)
$$

### Cálculo del ciclo de actividad (*duty cycle*)

$$
D = \frac{\tau_+}{T}
$$

donde $\tau_+$ es el valor promedio obtenido con los flancos $(+, -)$ y $T$ el período promedio obtenido con $(+, +)$.

### Propagación de incertidumbre del ciclo de actividad

$D$ es una magnitud indirecta. Aplicando la ley de propagación de la GUM:

$$
u_c(D) = \sqrt{\left( \frac{\partial D}{\partial \tau_+} \cdot u(\tau_+) \right)^2 + \left( \frac{\partial D}{\partial T} \cdot u(T) \right)^2}
$$

con los coeficientes de sensibilidad:

$$
c_{\tau_+} = \frac{\partial D}{\partial \tau_+} = \frac{1}{T} \qquad\qquad c_T = \frac{\partial D}{\partial T} = -\frac{\tau_+}{T^2}
$$

Como $\tau_+$ y $T$ se miden con la **misma base de tiempo**, su error está totalmente correlacionado y **se cancela en el cociente**: un error relativo $\varepsilon$ en $f_{BT}$ multiplica a numerador y denominador por $(1+\varepsilon)$. Por eso, para $D$ solo se combinan la Tipo A y el error de $\pm 1$ cuenta de cada intervalo:

$$
u_c(D) = \sqrt{\left( \frac{1}{T} \cdot u(\tau_+) \right)^2 + \left( \frac{\tau_+}{T^2} \cdot u(T) \right)^2}
\qquad , \qquad U(D) = k \cdot u_c(D)
$$

In [21]:
M_TI = 100     # multiplicador usado en T.I. A->B (x100)

# Mediciones [µs] - 10 muestras por combinación de flancos
datos_TI = {
    "20%": {
        "++": [1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069],
        "+-": [199.274, 199.275, 199.283, 199.280, 199.279, 199.274, 199.275, 199.283, 199.280, 199.279],
        "-+": [800.816, 800.818, 800.820, 800.823, 800.816, 800.818, 800.820, 800.823, 800.821, 800.822],
        "--": [1000.068, 1000.069, 1000.068, 1000.069, 1000.068, 1000.069, 1000.068, 1000.069, 1000.066, 1000.069],
    },
    "50%": {
        "++": [1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069],
        "+-": [500.052, 500.062, 500.052, 500.062, 500.052, 500.062, 500.052, 500.062, 500.052, 500.062],
        "-+": [500.041, 500.039, 500.053, 500.045, 500.041, 500.039, 500.053, 500.045, 500.039, 500.039],
        "--": [1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069, 1000.066, 1000.069],
    },
    "80%": {
        "++": [1000.062, 1000.064, 1000.062, 1000.064, 1000.062, 1000.064, 1000.062, 1000.064, 1000.062, 1000.064],
        "+-": [799.868, 799.864, 799.869, 799.868, 799.864, 799.869, 799.868, 799.864, 799.869, 799.864],
        "-+": [200.242, 200.240, 200.241, 200.242, 200.240, 200.241, 200.242, 200.240, 200.241, 200.240],
        "--": [1000.068, 1000.069, 1000.068, 1000.069, 1000.068, 1000.069, 1000.068, 1000.069, 1000.068, 1000.069],
    },
}

def medir_intervalo(mediciones_us):
    """Devuelve media, u_C (con base de tiempo), U y u sin base de tiempo (para cocientes)."""
    t = np.mean(mediciones_us)
    N = M_TI * t * 1e-6 * f_BT
    uA = incert_A(mediciones_us)
    uB = incert_B_ConMedia(error_rel_BT_pct, nd, N, t)     # base de tiempo + 1 cuenta
    uB_cta = incert_B_ConMedia(0, nd, N, t)                # solo 1 cuenta
    uC = incert_Total(uA, uB)
    return {'media': t, 'uA': uA, 'uB': uB, 'u_C': uC, 'U': k*uC,
            'u_sinBT': incert_Total(uA, uB_cta)}

TI = {d: {fl: medir_intervalo(v) for fl, v in fl_d.items()} for d, fl_d in datos_TI.items()}

def duty(tau, T):
    D = tau['media'] / T['media']
    uc_D = math.sqrt((1/T['media'] * tau['u_sinBT'])**2 + (tau['media']/T['media']**2 * T['u_sinBT'])**2)
    return D*100, k*uc_D*100      # en %

filas = []
for d in TI:
    T, tau = TI[d]["++"], TI[d]["+-"]
    D, U_D = duty(tau, T)
    Tm, TU = redondear(T['media'], T['U'])
    Dm, DU = redondear(D, U_D)
    filas.append([d, T['media'], T['uA'], T['uB'], T['U'], f"({Tm} ± {TU}) µs",
                  tau['media'], D, U_D, f"({Dm} ± {DU}) %"])
    resultados.append([f"Período T, duty {d}", "T.I. A→B (++) x100", f"({Tm} ± {TU}) µs"])
    resultados.append([f"Ciclo de actividad, duty {d}", "T.I. (+-)/(++) x100", f"({Dm} ± {DU}) %"])

df_ej2 = pd.DataFrame(filas, columns=["Duty nominal", "T [µs]", "uA(T) [µs]", "uB(T) [µs]", "U(T) [µs]",
                                      "Resultado T", "τ+ [µs]", "D [%]", "U(D) [%]", "Resultado D"])
pd.set_option("display.precision", 6)
print(df_ej2.to_string(index=False))

Duty nominal    T [µs]  uA(T) [µs]  uB(T) [µs]  U(T) [µs]           Resultado T  τ+ [µs]     D [%]  U(D) [%]            Resultado D
         20% 1000.0675    0.000500     0.01097   0.021963 (1000.067 ± 0.022) µs 199.2782 19.926475  0.000251 (19.92647 ± 0.00025) %
         50% 1000.0675    0.000500     0.01097   0.021963 (1000.067 ± 0.022) µs 500.0570 50.002325  0.000361 (50.00232 ± 0.00036) %
         80% 1000.0630    0.000333     0.01097   0.021951 (1000.063 ± 0.022) µs 799.8667 79.981631  0.000217 (79.98163 ± 0.00022) %


### Resultados

Valores medidos con sus incertidumbres expandidas ($k = 2$, nivel de confianza de aproximadamente 95%), señal rectangular de 1 kHz, $2\,V_{pp}$, T.I. A→B con multiplicador x100:

$$
\boxed{\text{Duty } 20\%:\quad T = (1000.067 \pm 0.022)\,\mu\text{s} \qquad D = (19.92647 \pm 0.00025)\,\%}
$$

$$
\boxed{\text{Duty } 50\%:\quad T = (1000.067 \pm 0.022)\,\mu\text{s} \qquad D = (50.00232 \pm 0.00036)\,\%}
$$

$$
\boxed{\text{Duty } 80\%:\quad T = (1000.063 \pm 0.022)\,\mu\text{s} \qquad D = (79.98163 \pm 0.00022)\,\%}
$$

## Extra: efecto de alternar los flancos de disparo

Se analizan las cuatro configuraciones de flancos $(+,+)$, $(-,-)$, $(+,-)$ y $(-,+)$. Si los flancos fueran ideales debería cumplirse:

* $T_{(++)} = T_{(--)}$
* $\tau_+ + \tau_- = T$
* el ciclo de actividad obtenido como $D_1 = \tau_+/T$ debería coincidir con $D_2 = 1 - \tau_-/T$.

In [22]:
filas_x = []
for d in TI:
    r = TI[d]
    fila = [d]
    for fl in ["++", "--", "+-", "-+"]:
        m, u = redondear(r[fl]['media'], r[fl]['U'])
        fila.append(f"{m} ± {u}")
    T = r["++"]['media']
    suma = r["+-"]['media'] + r["-+"]['media']
    D1 = r["+-"]['media'] / T * 100
    D2 = (1 - r["-+"]['media'] / T) * 100
    fila += [f"{(r['--']['media'] - T)*1e3:.1f}", f"{(suma - T)*1e3:.1f}", f"{D1:.4f}", f"{D2:.4f}"]
    filas_x.append(fila)

df_flancos = pd.DataFrame(filas_x, columns=["Duty", "T(++) [µs]", "T(--) [µs]", "τ+ (+-) [µs]", "τ- (-+) [µs]",
                                            "T(--)-T(++) [ns]", "τ+ + τ- - T [ns]", "D1=τ+/T [%]", "D2=1-τ-/T [%]"])
print(df_flancos.to_string(index=False))

Duty       T(++) [µs]       T(--) [µs]      τ+ (+-) [µs]      τ- (-+) [µs] T(--)-T(++) [ns] τ+ + τ- - T [ns] D1=τ+/T [%] D2=1-τ-/T [%]
 20% 1000.067 ± 0.022 1000.068 ± 0.022 199.2782 ± 0.0057   800.820 ± 0.018              0.8             30.4     19.9265       19.9234
 50% 1000.067 ± 0.022 1000.067 ± 0.022   500.057 ± 0.012   500.043 ± 0.012              0.0             32.9     50.0023       49.9990
 80% 1000.063 ± 0.022 1000.068 ± 0.022   799.867 ± 0.018 200.2409 ± 0.0053              5.5             44.6     79.9816       79.9772


**Análisis:**

* **$(+,+)$ y $(-,-)$** dan el mismo período dentro de la incertidumbre (diferencias de 0 a 5 ns sobre 1 ms). Al medir entre flancos **del mismo tipo**, cualquier retardo de disparo (nivel de trigger distinto del 50%, tiempo de subida o bajada finito, histéresis del comparador, diferencia de retardo entre canales) afecta igual a la apertura y al cierre de la compuerta y **se cancela**.
* **$(+,-)$ y $(-,+)$** miden entre flancos **de distinto tipo**, y ahí esos errores no se cancelan. Por eso $\tau_+ + \tau_-$ no reproduce exactamente $T$: la suma difiere en unos 30 ns, 33 ns y 45 ns para 20%, 50% y 80%. Es decir, cada combinación mixta incluye un error sistemático de decenas de ns que la incertidumbre calculada (base de tiempo y ±1 cuenta) no contempla. El error depende de la pendiente de los flancos y de la posición del nivel de disparo: si el nivel no está exactamente a mitad de la excursión, el flanco de subida se detecta "tarde" y el de bajada "temprano", o al revés.
* Por esto, el ciclo de actividad calculado como $\tau_+/T$ y como $1-\tau_-/T$ difiere en unas milésimas de % (0.003% a 0.004%), bastante más que la incertidumbre calculada para $D$. Para una medición más exacta habría que ajustar el *trigger level* al 50% de la señal y usar una señal con flancos rápidos. El período, en cambio, conviene medirlo siempre entre flancos iguales.

# Ejercicio 3

Se mide la relación de frecuencia entre una señal senoidal de 10 MHz y una de 1 MHz utilizando el modo de medición de relación de frecuencias (Ratio A/B) con la mayor resolución posible.

### Medición de relación de frecuencias (Ratio A/B)

En este modo la señal del canal B reemplaza a la base de tiempo: la compuerta queda abierta durante $M$ períodos de la señal B (multiplicador x1, x10, x100 o x1000), y durante ese tiempo se cuentan los $N_A$ ciclos de la señal A:

$$
R = \frac{f_A}{f_B} = \frac{N_A}{M} \qquad \Rightarrow \qquad N_A = R \cdot M
$$

La **mayor resolución** se obtiene con el multiplicador máximo, $M = 1000$, que es el usado: la resolución resulta $1/M = 0.001$, coherente con las tres cifras decimales de las lecturas.

#### Evaluación de la incertidumbre (GUM)

Como la compuerta la genera la propia señal B y no la base de tiempo interna, **el error de la base de tiempo no interviene**. Quedan:

1. **Incertidumbre Tipo A:** a partir de las 10 lecturas, $u_A = s(R)/\sqrt{n}$.
2. **Incertidumbre Tipo B:** el error de $\pm 1$ cuenta en $N_A$, con distribución rectangular:

$$
u_B = \frac{1}{\sqrt{3}}\cdot\frac{\Delta N}{N_A}\cdot R = \frac{1}{\sqrt{3}\,M}
$$

que se calcula con `incert_B_ConMedia(0, nd, R·M, R)`, con error de base de tiempo igual a 0.

In [23]:
M_ratio = 1000      # multiplicador x1000 (máxima resolución)

def medir_ratio(mediciones, nombre):
    R = np.mean(mediciones)
    N_A = R * M_ratio
    uA = incert_A(mediciones)
    uB = incert_B_ConMedia(0, nd, N_A, R)
    uC = incert_Total(uA, uB)
    U = k * uC
    print(f"--- {nombre} ---   (N_A = {N_A:.0f} cuentas)")
    imprimir("R", "", R, uA, uB, uC, U)
    print()
    return {'media': R, 'u_C': uC, 'U': U}

# Relación 1 MHz / 10 MHz
R_1_10 = medir_ratio([0.100]*10, "Ratio 1 MHz / 10 MHz")

# Relación 10 MHz / 1 MHz
R_10_1 = medir_ratio([13.123, 12.987, 13.123, 12.987, 13.123, 12.987, 13.123, 12.987, 13.123, 12.987],
                     "Ratio 10 MHz / 1 MHz")

for nom, r in [("Relación 1 MHz / 10 MHz", R_1_10), ("Relación 10 MHz / 1 MHz (anómala)", R_10_1)]:
    m, u = redondear(r['media'], r['U'])
    resultados.append([nom, "Ratio A/B x1000", f"{m} ± {u}"])

--- Ratio 1 MHz / 10 MHz ---   (N_A = 100 cuentas)
Media = 0.1 
uA = 0.000e+00 
uB = 5.774e-04 
uC = 5.774e-04 
U (k=2) = 1.155e-03 
Resultado: R = (0.1000 ± 0.0012) 

--- Ratio 10 MHz / 1 MHz ---   (N_A = 13055 cuentas)
Media = 13.055 
uA = 2.267e-02 
uB = 5.774e-04 
uC = 2.267e-02 
U (k=2) = 4.535e-02 
Resultado: R = (13.055 ± 0.045) 



### Resultados

Relación 1 MHz / 10 MHz:

$$
\boxed{R_{1/10} = 0.1000 \pm 0.0012}
$$

Relación 10 MHz / 1 MHz:

$$
\boxed{R_{10/1} = 13.055 \pm 0.045}
$$

*Incertidumbre expandida con $k = 2$ (nivel de confianza de aproximadamente 95%), señales senoidales, multiplicador x1000.*

La relación **1 MHz / 10 MHz** da exactamente el valor esperado, $0.100$, con lecturas totalmente estables.

La relación **10 MHz / 1 MHz**, en cambio, no da el valor esperado de 10. Durante la práctica observamos que **la señal del canal B no valía 10 MHz sino aproximadamente 12 MHz, porque la compuerta está usando el mismo clock para la medición y nosotros lo estábamos reinsertando desde el propio instrumento**. Al tomar la señal de referencia del propio contador y volver a ingresarla, la medición deja de ser independiente y la lectura no corresponde a la señal que se quería medir. Además, la lectura alterna entre 13.123 y 12.987, lo que muestra que el conteo no estaba sincronizado de forma estable. Por eso la incertidumbre Tipo A domina casi por completo y este resultado se informa solo como registro de la anomalía, no como la relación buscada.

## Extra: Ratio A/B frente al cociente de mediciones directas

No se registraron lecturas directas en modo frecuencia de las senoidales de 10 MHz y 1 MHz, por lo que la comparación se hace en términos de la **resolución** de cada método, con valores nominales. Como referencia, en el Ejercicio 1 la señal de 1 MHz medida con gate de 10 s se leyó con 0.1 Hz de resolución.

**Cociente de mediciones directas:** $R = f_A / f_B$, cada una medida en modo frecuencia con $T_C = 10\,\text{s}$:

$$
\frac{u(R)}{R} = \sqrt{\left(\frac{u(f_A)}{f_A}\right)^2 + \left(\frac{u(f_B)}{f_B}\right)^2 - 2\,r\,\frac{u(f_A)}{f_A}\frac{u(f_B)}{f_B}}
$$

La parte de base de tiempo es la misma en ambas mediciones ($r = 1$), así que se cancela, y solo queda la de $\pm 1$ cuenta, $\dfrac{1}{\sqrt{3}\,f\,T_C}$. Se usa el helper `incert_Comb_Rel` con `correlacion = 1` para la base de tiempo.

**Ratio A/B:** la resolución está fijada por el multiplicador, $\dfrac{\Delta R}{R} = \dfrac{1}{N_A} = \dfrac{1}{R\,M}$.

In [24]:
f_10M, f_1M, T_C = 10e6, 1e6, 10

# Componente de base de tiempo (correlacionada r=1): se cancela en el cociente
u_BT_10M = {'media': f_10M, 'u_C': error_rel_BT * f_10M / math.sqrt(3)}
u_BT_1M  = {'media': f_1M,  'u_C': error_rel_BT * f_1M  / math.sqrt(3)}
urel_BT = incert_Comb_Rel(u_BT_10M, u_BT_1M, correlacion=1)

# Componente de ±1 cuenta (independiente)
u_cta_10M = {'media': f_10M, 'u_C': incert_B_ConMedia(0, nd, f_10M*T_C, f_10M)}
u_cta_1M  = {'media': f_1M,  'u_C': incert_B_ConMedia(0, nd, f_1M*T_C, f_1M)}
urel_cta = incert_Comb_Rel(u_cta_10M, u_cta_1M, correlacion=0)

urel_div = math.sqrt(urel_BT**2 + urel_cta**2)

print("Cociente de mediciones directas (gate 10 s):")
print(f"  resolución f(10 MHz) = {1/T_C} Hz  -> {1/(f_10M*T_C):.1e} relativa")
print(f"  resolución f(1 MHz)  = {1/T_C} Hz  -> {1/(f_1M*T_C):.1e} relativa")
print(f"  u_rel por base de tiempo (r=1) = {urel_BT:.1e}  (se cancela)")
print(f"  u_rel del cociente             = {urel_div:.2e}")
print()
print("Ratio A/B con multiplicador x1000:")
for R in [10, 0.1]:
    print(f"  R = {R:>4}: N_A = {R*M_ratio:>6.0f}  ->  resolución 1/N_A = {1/(R*M_ratio):.0e} relativa"
          f",  u_rel = {1/(math.sqrt(3)*R*M_ratio):.2e}")
print()
print(f"Mejora en resolución del cociente frente al Ratio (R = 10): {(1/(math.sqrt(3)*10*M_ratio))/urel_div:.0f} veces")

Cociente de mediciones directas (gate 10 s):
  resolución f(10 MHz) = 0.1 Hz  -> 1.0e-08 relativa
  resolución f(1 MHz)  = 0.1 Hz  -> 1.0e-07 relativa
  u_rel por base de tiempo (r=1) = 0.0e+00  (se cancela)
  u_rel del cociente             = 5.80e-08

Ratio A/B con multiplicador x1000:
  R =   10: N_A =  10000  ->  resolución 1/N_A = 1e-04 relativa,  u_rel = 5.77e-05
  R =  0.1: N_A =    100  ->  resolución 1/N_A = 1e-02 relativa,  u_rel = 5.77e-03

Mejora en resolución del cociente frente al Ratio (R = 10): 995 veces


**Conclusión del extra:** con este contador, **el cociente de las mediciones directas con gate de 10 s da una resolución mucho mejor** (del orden de $10^{-7}$) que el modo Ratio A/B, limitado a $10^{-4}$ con $R = 10$ y a $10^{-2}$ con $R = 0.1$ por el multiplicador máximo x1000. El modo Ratio cuenta como máximo $R \cdot 1000$ ciclos, mientras que la medición directa con 10 s cuenta $10^7$ a $10^8$ ciclos.

La ventaja del modo Ratio es que es una sola medición, simultánea sobre ambas señales, e independiente de la base de tiempo del contador. Para el cociente de mediciones directas, la base de tiempo también se cancela solo si ambas frecuencias se miden con el mismo instrumento y en condiciones estables. Además, para que el Ratio sea correcto hay que poner en el canal A la señal de mayor frecuencia, que es la que más cuentas aporta.

# Resultados de la medición

Todos los resultados con su incertidumbre expandida ($k = 2$, nivel de confianza de aproximadamente 95%). Contador Protek U2000A, base de tiempo de 10 MHz con 18 ppm (antigüedad supuesta de 5 años), señales de $2\,V_{pp}$.

| Mensurando | Condición | Resultado |
| :--- | :--- | :---: |
| Frecuencia 10 Hz | Gate 0.1 s (señal cuadrada) | (10 ± 12) Hz |
| Frecuencia 10 Hz | Gate 1 s (señal cuadrada) | (10.0 ± 1.2) Hz |
| Frecuencia 10 Hz | Gate 10 s (señal cuadrada) | (10.00 ± 0.12) Hz |
| Período 10 Hz | M = 1 (señal cuadrada) | (100002.9 ± 2.2) µs |
| Período 10 Hz | M = 10 (señal cuadrada) | (100003.2 ± 2.1) µs |
| Frecuencia 1 kHz | Gate 0.1 s (señal cuadrada) | (1000 ± 12) Hz |
| Frecuencia 1 kHz | Gate 1 s (señal cuadrada) | (1000.0 ± 1.2) Hz |
| Frecuencia 1 kHz | Gate 10 s (señal cuadrada) | (1000.00 ± 0.14) Hz |
| Período 1 kHz | M = 1 (señal cuadrada) | (1000.05 ± 0.14) µs |
| Período 1 kHz | M = 10 (señal cuadrada) | (1000.035 ± 0.033) µs |
| Período 1 kHz | M = 100 (señal cuadrada) | (1000.035 ± 0.022) µs |
| Frecuencia 1 MHz | Gate 0.1 s (señal cuadrada) | (999965 ± 33) Hz |
| Frecuencia 1 MHz | Gate 1 s (señal cuadrada) | (999965 ± 22) Hz |
| Frecuencia 1 MHz | Gate 10 s (señal cuadrada) | (999965 ± 21) Hz |
| Período 1 MHz | M = 1 (señal cuadrada) | (1.00 ± 0.12) µs |
| Período 1 MHz | M = 10 (señal cuadrada) | (1.000 ± 0.012) µs |
| Período 1 MHz | M = 100 (señal cuadrada) | (1.0000 ± 0.0012) µs |
| Período 10 Hz | M = 100 (señal cuadrada) | Overflow: no medible |
| Período T, duty 20% | T.I. A→B (++) x100 | (1000.067 ± 0.022) µs |
| Ciclo de actividad, duty 20% | T.I. (+-)/(++) x100 | (19.92647 ± 0.00025) % |
| Período T, duty 50% | T.I. A→B (++) x100 | (1000.067 ± 0.022) µs |
| Ciclo de actividad, duty 50% | T.I. (+-)/(++) x100 | (50.00232 ± 0.00036) % |
| Período T, duty 80% | T.I. A→B (++) x100 | (1000.063 ± 0.022) µs |
| Ciclo de actividad, duty 80% | T.I. (+-)/(++) x100 | (79.98163 ± 0.00022) % |
| Relación 1 MHz / 10 MHz | Ratio A/B x1000 | 0.1000 ± 0.0012 |
| Relación 10 MHz / 1 MHz (anómala) | Ratio A/B x1000 | 13.055 ± 0.045 |

# Conclusiones

* **Resolución en modo frecuencia y en modo período.** Cada vez que el gate time o el multiplicador de períodos crecen 10 veces, la resolución mejora 10 veces porque $N$ aumenta en esa proporción. A **baja frecuencia** el modo frecuencia es muy pobre: a 10 Hz con 0.1 s se cuenta 1 solo ciclo y la incertidumbre es del orden del 100%. Ahí conviene el **modo período**, que cuenta los pulsos de 10 MHz de la base de tiempo, con $10^6$ cuentas en un solo período. A **alta frecuencia** ocurre lo contrario: a 1 MHz el período tiene solo 10 cuentas por ciclo y conviene el **modo frecuencia**. El punto de cruce entre ambos métodos está en $f = \sqrt{f_{BT}/T_C}$, alrededor de 3 kHz con 1 s de gate.
* **Límite de la base de tiempo.** Cuando $N$ es grande, el término $\Delta N/N$ se vuelve despreciable y la incertidumbre queda dominada por la base de tiempo (18 ppm). Por eso a 1 MHz, a partir de 1 s de gate, la incertidumbre expandida queda prácticamente fija ($\approx 21$ a $22$ Hz, contra 33 Hz con 0.1 s), aunque la resolución siga mejorando 10 veces: **más resolución no implica más exactitud**.
* **Generador.** La señal de 1 MHz midió $\approx 999965$ Hz, unos 35 ppm por debajo del valor nominal. La desviación supera la incertidumbre del contador, así que se atribuye principalmente a la inexactitud del generador de funciones. Lo mismo se observa en los períodos de 10 Hz (100003 µs, +30 ppm) y de 1 kHz.
* **Overflow.** El período de 10 Hz promediado 100 veces no pudo medirse: la lectura requería 9 dígitos y el display tiene 8.
* **Cuadrada frente a senoidal (Extra 1).** Con senoidal la indicación es menos estable por el error de disparo: la pendiente en el cruce del nivel de trigger es baja y el ruido se convierte en jitter temporal. El error relativo estimado es de unos 300 ppm por período, contra un valor despreciable para la cuadrada. Promediar más períodos lo reduce en $1/M$.
* **Ciclo de actividad (Ejercicio 2 y Extra 2).** Los ciclos de actividad medidos fueron de aproximadamente 19.93%, 50.00% y 79.98%, cercanos a los nominales. En el cociente $\tau_+/T$ la base de tiempo se cancela, por eso la incertidumbre del ciclo de actividad es muy chica. Las combinaciones de flancos iguales (++ y --) dan el mismo período, porque los retardos de disparo se cancelan. En las mixtas (+- y -+) no se cancelan y aparece un error sistemático de decenas de ns ($\tau_+ + \tau_- \neq T$). Para medir anchos de pulso hay que ajustar el nivel de disparo al 50% de la señal.
* **Relación de frecuencias (Ejercicio 3 y Extra 3).** La relación 1 MHz / 10 MHz con multiplicador x1000 dio $0.1000 \pm 0.0012$, como se esperaba. La relación 10 MHz / 1 MHz resultó anómala, $\approx 13$ en lugar de 10, porque la señal del canal B era la propia referencia del contador reinsertada. En resolución, el cociente de mediciones directas con gate de 10 s ($\sim 10^{-7}$) supera ampliamente al modo Ratio ($10^{-4}$ como máximo con $R = 10$). La ventaja del Ratio es que no depende de la base de tiempo y entrega la relación en una sola medición.